# Numerical inverse images: parameters, matrix and masses

Show a completely traceable real example from the archived data, then compare three restart solutions for one and the same target.

**Data:** the frozen 3000-run production set (`data/raw`). **Acceptance:** all five normalized residual magnitudes strictly below one. These are *numerical widths*, not independently measured Gaussian standard deviations.

> Start from the repository root or its `notebooks/` folder. No recomputation of the 3000 DE optimizations is required.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works in Jupyter, VS Code, and Colab if the repository is cloned locally.
workdir = Path.cwd().resolve()
candidates = [workdir, workdir.parent]
ROOT = next((p for p in candidates if (p / "data/raw/runs.csv").is_file()
             and (p / "src/model.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(
        "Cannot find the repository root. Open the project folder (with data/ and src/) "
        "or clone the repository in the Colab session before running the notebook."
    )
sys.path.insert(0, str(ROOT / "src"))
from model import PARAMETERS, V_EW, MASS_WIDTHS, mass_matrix, spectrum
raw = pd.read_csv(ROOT / "data/raw/runs.csv")
print(f"Repository: {ROOT.name}; records: {len(raw):,}; v = {V_EW} GeV")

Repository: DE_Particle_paper_public; records: 3,000; v = 246.22 GeV


In [2]:
residual_cols = ["z_H1", "z_H2", "z_H3", "z_charged", "z_kappa"]
z_abs = np.abs(raw[residual_cols].to_numpy(float))
mask_mass = (z_abs[:, :3] < 1).all(axis=1)
mask_charged = z_abs[:, 3] < 1
mask_kappa = z_abs[:, 4] < 1
mask_all = (z_abs < 1).all(axis=1)
accepted = raw.loc[mask_all].copy()
representative = (accepted.sort_values(["target_index", "loss", "restart"])
                  .drop_duplicates("target_index").sort_values("target_index"))
assert len(raw) == 3000 and len(accepted) == 2971 and len(representative) == 1000

## Example from target 0, restart 2\nThe ten input parameters are reported in the exact order used by the frozen forward model. R contains mass eigenvectors as columns.

In [3]:
row=raw[(raw.target_index==0)&(raw.restart==2)].iloc[0]
x=row[list(PARAMETERS)].to_numpy(float)
M2=mass_matrix(x)
eigenvalues,R=np.linalg.eigh(M2)
if (row.v_1*R[0,1] + np.sqrt(V_EW*V_EW-row.v_1**2)*R[1,1])<0:R[:,1]*=-1
masses=np.sqrt(eigenvalues)
targets=row[["target_H1","target_H2","target_H3"]].to_numpy(float)
print("Parameters (GeV for v1, v_sigma, mu):")
print(pd.Series(x,index=PARAMETERS).to_string(float_format=lambda q:f"{q:.12g}"))
print("\nM_H^2 [GeV^2]:")
print(pd.DataFrame(M2).to_string(index=False,header=False,float_format=lambda q:f"{q: .6f}"))
check=pd.DataFrame({"target_GeV":targets,"predicted_GeV":masses,"abs_residual_over_sigma":np.abs((masses-targets)/MASS_WIDTHS)},index=["H1","H2","H3"])
print("\n",check.to_string(float_format=lambda q:f"{q:.9g}"))
print("\nR^T M² R - diag(m²), maximum abs (GeV²):",np.abs(R.T@M2@R-np.diag(eigenvalues)).max())
print("Charged mass (GeV):",row.mc,"kappa2V:",row.kappa2V)
assert check.abs_residual_over_sigma.max()<1

Parameters (GeV for v1, v_sigma, mu):
lambda_1          0.502687153735
lambda_2          0.131085305736
lambda_3          0.226746528581
lambda_4         -0.255602039531
lambda_sigma      0.143033896384
lambda_1sigma     0.680962576066
lambda_2sigma   -0.0190686874877
v_1               0.566770876569
v_sigma            281.352943505
mu               -0.105096440488

M_H^2 [GeV^2]:
 9083.528099    -24.935362     90.290455
  -24.935362  15893.870672  -1321.016639
   90.290455  -1321.016639  22645.014242

     target_GeV  predicted_GeV  abs_residual_over_sigma
H1   95.304188     95.3041867           2.53295619e-05
H2  125.078362     125.078364           2.07241445e-05
H3  151.310701       151.3107           1.46661216e-05

R^T M² R - diag(m²), maximum abs (GeV²): 1.4551915228366852e-11
Charged mass (GeV): 129.73472608593752 kappa2V: 0.9826481755258528


## Same target, different parameter vectors\nCompare the VEV and couplings across the three *independent restarts*. Do not count these as three experimental measurements.

In [4]:
same=raw[raw.target_index==0].sort_values("restart")
print(same[["restart","seed","loss","lambda_sigma","v_sigma","v_1","mu","mc","kappa2V"]].to_string(index=False,float_format=lambda q:f"{q:.8g}"))
assert len(same)==3

 restart  seed          loss  lambda_sigma   v_sigma        v_1          mu        mc    kappa2V
       0    11 2.4527706e-09    0.13686682  289.1846  3.7978862 -0.68411431 130.06614  0.9992528
       1    23 1.9323854e-09    0.10532447   328.975  3.5042866 -0.56073933 129.94196 0.99806264
       2    37  1.286172e-09     0.1430339 281.35294 0.56677088 -0.10509644 129.73473 0.98264818


## A counterexample to a universal singlet-dominance approximation\nThis archived parameter set has a different eigenstate composition; the approximation \(2\lambda_\sigma v_\sigma^2\simeq m_{H_3}^2\) is **not** applicable without testing the singlet component.

In [5]:
branch=raw[(raw.target_index==145)&(raw.restart==1)].iloc[0]
x2=branch[list(PARAMETERS)].to_numpy(float)
w2,R2=np.linalg.eigh(mass_matrix(x2))
print("Singlet weight in (H1,H2,H3):",np.round(R2[2,:]**2,5))
print("m_H3^2 (GeV²):",w2[2],"2 lambda_sigma v_sigma^2 (GeV²):",2*branch.lambda_sigma*branch.v_sigma**2)

Singlet weight in (H1,H2,H3): [0.99595 0.00288 0.00117]
m_H3^2 (GeV²): 22978.176317159858 2 lambda_sigma v_sigma^2 (GeV²): 9132.596325147988
